## Gradio Pets

In [ ]:
from fastai.vision.all import *
import timm

In [ ]:
path = untar_data(URLs.PETS)/'images'

dls = ImageDataLoaders.from_name_func('.',
    get_image_files(path), valid_pct=0.2, seed=42,
    label_func=RegexLabeller(pat = r'^([^/]+)_\d+'),
    item_tfms=Resize(224))

In [ ]:
dls.show_batch(max_n=4)

In [ ]:
print(path)

In [ ]:
!find /root/.fastai/data/oxford-iiit-pet/images | wc -l

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(3)

In [ ]:
learn.export('resnet34_0_3.pkl')  # good idea to save your experiments 

In [ ]:
learn.fine_tune(1)  
# something is wrong!!! 
# We will learn why in later lessons. Hint: lr

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(4)

In [ ]:
# compare outputs of learn.fine_tune(4) learn.fine_tune(3)
# Do you see anything interesting?

In [ ]:
# Let's add augmentations from Lesson-2
# item_tfms=RandomResizedCrop(224, min_scale=0.5) 
# batch_tfms=aug_transforms()

dls2 = ImageDataLoaders.from_name_func('.',
    get_image_files(path), valid_pct=0.2, seed=42,
    label_func=RegexLabeller(pat = r'^([^/]+)_\d+'),
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms(),                      
                                     )

In [ ]:
dls2.show_batch(max_n=8)

In [ ]:
learn2 = vision_learner(dls2, resnet34, metrics=error_rate)
learn2.fine_tune(4)

In [ ]:
# Compare the outputs of learn2.fine_tune(4) and learn.fine_tune(4)
# ...

We could try a better model, based on [this analysis](https://www.kaggle.com/code/jhoward/which-image-models-are-best/). The convnext models work great!

In [ ]:
timm.list_models('convnext*')

In [ ]:
# learn = vision_learner(dls, 'convnext_tiny_in22k', metrics=error_rate).to_fp16()
learn = vision_learner(dls, 'convnext_tiny', metrics=error_rate)
learn.fine_tune(3)

In [ ]:
learn.export('model.pkl')

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(3)

## Model Comparison

In this experiment, I compared ResNet34 and ConvNeXtV2_nano on the Pets dataset.

ResNet34 achieved an error rate of 0.0636 (≈93.6% accuracy).
ConvNeXtV2_nano achieved an error rate of 0.0575 (≈94.2% accuracy).

The ConvNeXtV2_nano model performed slightly better than ResNet34.
This shows that newer architectures can provide better performance, 
but the difference is not extremely large.

This experiment demonstrates how model architecture influences performance.